# NLA Studio on Colab (qwen7b)

Runs the released **kitft Qwen2.5-7B NLA** end to end: base-model extraction, the AV (vector to text) on **SGLang/GPU**, the AR (text to vector) on **CPU**. The code is pulled with `git clone` from `nla-studio` on your fork, so the only file you manage is this notebook.

**Set the runtime first:** Runtime -> Change runtime type -> GPU **A100** (Pro+) or **L4** (Pro), shape **High-RAM**.
The free **T4 will not fit**. qwen7b is ungated, so no token is needed.

Run the cells top to bottom.


In [ ]:
!nvidia-smi


## 1. Get the code (git clone)
Clones `nla_studio/` from your fork. Re-run this cell to pull the latest after any push.


In [ ]:
!rm -rf /content/natural_language_autoencoders
!git clone --depth 1 -b nla-studio https://github.com/agastyasridharan/natural_language_autoencoders.git /content/natural_language_autoencoders
%cd /content/natural_language_autoencoders/nla_studio
!ls


## 2. Install dependencies (~5 min)
Installs the **pinned** stack: `sglang[all]==0.5.6` + `transformers==4.57.1` + `torch==2.9.1`.


In [ ]:
!pip install -q -r requirements.txt


## 3. Preflight: verify the tokenizer stack (seconds, no GPU)
Downloads only the AV tokenizer + sidecar and runs the exact gate `/load_family` uses. If it **FAILs**
with "injection token appears 0x", transformers got upgraded; run the printed fix, then
**Runtime -> Restart session** and re-run from cell 2.


In [ ]:
import transformers, tokenizers
print("transformers", transformers.__version__, "| tokenizers", tokenizers.__version__)
!python scripts/preflight_tokenizer.py --family qwen7b


## 4. Launch the AV (SGLang) server on the GPU
Downloads the 7B AV (~15 GB) and loads it; waits until healthy.


In [ ]:
import subprocess, time, os, httpx
sg = subprocess.Popen(["bash", "scripts/launch_sglang.sh", "qwen7b"],
                      stdout=open("sglang.log", "w"), stderr=subprocess.STDOUT)
ok = False
for _ in range(240):                      # ~20 min cap (download + load)
    try:
        httpx.get("http://localhost:30000/get_model_info", timeout=5); ok = True; break
    except Exception:
        if sg.poll() is not None:
            print("SGLang exited early - see log below"); break
        time.sleep(5)
print("AV up" if ok else "AV NOT up")
!tail -n 20 sglang.log


## 5. Start the web app (base + AR on CPU)
`NLA_INPROC_DEVICE=cpu` keeps the single GPU free for the AV.


In [ ]:
os.environ["NLA_INPROC_DEVICE"] = "cpu"
os.environ["NLA_SGLANG_URL"]    = "http://localhost:30000"
uv = subprocess.Popen(["python", "-m", "uvicorn", "app.server:app", "--host", "0.0.0.0", "--port", "8000"],
                      stdout=open("uvicorn.log", "w"), stderr=subprocess.STDOUT)
time.sleep(6)
!tail -n 20 uvicorn.log


## 6. Open the UI
Opens NLA Studio via Colab's port proxy. In the UI: **Load** `qwen7b` (first load pulls base+AR into
CPU RAM, ~1-3 min), then **Extract** -> **Explain (AV)** -> **Score**.


In [ ]:
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8000)


## Troubleshooting
- **"injection token appears 0x"**: transformers got upgraded past 4.57.x. Run
  `!pip install "sglang[all]==0.5.6" "transformers==4.57.1"`, then **Runtime -> Restart session**, re-run from cell 2.
- **OOM on the GPU**: kill SGLang and relaunch smaller: `!NLA_MEM_FRACTION=0.7 bash scripts/launch_sglang.sh qwen7b`.
- **Extract is slow**: 7B forward on CPU (~10-60 s per token click); keep text short, use the suggested token.
- **gemma / llama**: set `os.environ["HF_TOKEN"]="hf_..."` before cells 2-4 (gated). **llama70b will not fit one Colab GPU.**
- Logs on demand: run the cell below.


In [ ]:
!tail -n 60 sglang.log
print("\n----- uvicorn -----")
!tail -n 60 uvicorn.log
